In [1]:
# ============================================================
# STATISTICAL ANALYSIS FOR THE EEG-FORMER PAPER
# Robust version:
# - Searches for any CSV file inside each model folder
# - Selects the CSV containing seed, fold, and metric columns
# - Performs Friedman + paired Wilcoxon + Holm correction
# - Generates an article-style table against EEG-Former:
#   Model | Avg. Rank | Holm-corrected p-value | Significant
# - Saves the LaTeX table to file, but does NOT print it
# ============================================================

import os
import re
import itertools
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon, friedmanchisquare

try:
    from IPython.display import display
except ImportError:
    display = print


# ============================================================
# 1) GENERAL CONFIGURATION
# ============================================================

TARGET_MODEL = "EEG-Former"
ALPHA = 0.05
EXPECTED_N_PAIRS = 50  # 10 seeds x 5 folds

MODEL_PATHS = {
    "CNN-LSTM EEGNet": "/kaggle/input/datasets/alejandragomezr/seed-cnn-lstm/results_cnn_lstm_eegnet_10seeds",
    "T-GARNet": "/kaggle/input/datasets/alejandragomezr/seed-danna/results_tgarnet_10seeds",
    "EEGNet": "/kaggle/input/datasets/alejandragomezr/seed-eegnet/results_eegnet_10seeds",
    "Multi-Stream Transformer": "/kaggle/input/datasets/alejandragomezr/seed-multistream/Results_multistream 10 seed",
    "ShallowConvNet": "/kaggle/input/datasets/alejandragomezr/seed-shallow/results_shallowconvnet_10seeds",
    "IM--CBGT": "/kaggle/input/datasets/alejandragomezr/seed-im-cbgt/results_imcbgt_10seeds",
    "EEG-Former": "/kaggle/input/datasets/alejandragomezr/seed-eeg-former/results_ourmodel_10seeds",
}

OUT_DIR = "/kaggle/working/statistical_tests_EEGFormer_article"
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# 2) METRICS
# ============================================================

METRIC_CANDIDATES = {
    "accuracy": [
        "accuracy", "acc", "test_accuracy", "test_acc"
    ],
    "balanced_accuracy": [
        "balanced_acc", "balanced_accuracy", "bacc",
        "test_balanced_acc", "test_balanced_accuracy", "test_bacc"
    ],
    "recall": [
        "recall", "sensitivity", "test_recall", "test_sensitivity"
    ],
    "precision": [
        "precision", "test_precision"
    ],
    "f1": [
        "f1", "f1_score", "test_f1", "test_f1_score"
    ],
    "auc": [
        "auc", "roc_auc", "test_auc", "test_roc_auc"
    ],
}

# For the paper, the main statistical test is performed on accuracy.
STATISTICAL_METRICS = ["accuracy"]


# ============================================================
# 3) HELPER FUNCTIONS
# ============================================================

def clean_columns(df):
    """
    Clean column names to make them easier to detect.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(".", "_", regex=False)
    )
    return df


def find_first_existing_column(df, possible_names):
    """
    Return the first column name that exists in the dataframe.
    """
    for name in possible_names:
        if name in df.columns:
            return name
    return None


def normalize_series_to_percent(values):
    """
    Convert values to percentage if they are in the [0, 1] scale.
    If they are already in the [0, 100] scale, keep them unchanged.
    """
    values = np.asarray(values, dtype=float)

    if np.all(np.isnan(values)):
        return values

    max_val = np.nanmax(values)

    if max_val <= 1.5:
        return values * 100.0

    return values


def normalize_wide_to_percent(wide, model_names):
    """
    Convert each model column to percentage when needed.
    """
    wide = wide.copy()

    for model in model_names:
        wide[model] = normalize_series_to_percent(
            wide[model].astype(float).values
        )

    return wide


def holm_bonferroni_adjust(p_values):
    """
    Holm-Bonferroni correction for multiple comparisons.
    """
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full_like(p_values, np.nan, dtype=float)

    valid_mask = ~np.isnan(p_values)
    valid_p = p_values[valid_mask]

    m = len(valid_p)

    if m == 0:
        return adjusted

    order = np.argsort(valid_p)
    sorted_p = valid_p[order]

    sorted_adj = np.empty(m, dtype=float)
    running_max = 0.0

    for i, p in enumerate(sorted_p):
        adj_p = (m - i) * p
        running_max = max(running_max, adj_p)
        sorted_adj[i] = min(running_max, 1.0)

    valid_adjusted = np.empty(m, dtype=float)
    valid_adjusted[order] = sorted_adj
    adjusted[valid_mask] = valid_adjusted

    return adjusted


def format_p_display(p):
    """
    Format p-values for display in Python.
    Example: 9.33e-01, 5.05e-04.
    """
    if pd.isna(p):
        return "---"
    return f"{p:.2e}"


def format_p_latex(p):
    """
    Format p-values in homogeneous LaTeX scientific notation.
    Example: 9.33 x 10^{-1}, 5.05 x 10^{-4}.
    """
    if pd.isna(p):
        return "---"

    mantissa, exponent = f"{p:.2e}".split("e")
    mantissa = float(mantissa)
    exponent = int(exponent)

    return f"${mantissa:.2f} \\times 10^{{{exponent}}}$"


def list_all_csv_files(model_dir):
    """
    List all CSV files inside a model folder.
    """
    csv_files = []

    if not os.path.exists(model_dir):
        raise FileNotFoundError(
            f"The folder does not exist:\n{model_dir}\n"
            "Please check that the dataset is attached to the Kaggle notebook."
        )

    for root, _, files in os.walk(model_dir):
        for f in files:
            if f.lower().endswith(".csv"):
                csv_files.append(os.path.join(root, f))

    csv_files = sorted(csv_files)
    return csv_files


def infer_seed_from_path(path):
    """
    Try to infer the seed from the file or folder name.
    Valid examples:
    seed_123, seed-123, seed123, random_seed_42.
    """
    text = path.lower()

    patterns = [
        r"seed[_\- ]?(\d+)",
        r"random[_\- ]?seed[_\- ]?(\d+)",
        r"run[_\- ]?(\d+)"
    ]

    for pat in patterns:
        match = re.search(pat, text)
        if match:
            return int(match.group(1))

    return None


# ============================================================
# 4) CSV SELECTION AND LOADING
# ============================================================

def select_best_csv_for_model(model_name, model_dir):
    """
    Select the best CSV file for a given model.

    Criteria:
    1. It should contain a fold column.
    2. It should contain a seed column, or the seed should be inferable from the path.
    3. It should contain at least one known metric column.
    4. Files with more useful rows are preferred.
    """
    csv_files = list_all_csv_files(model_dir)

    if len(csv_files) == 0:
        raise FileNotFoundError(
            f"No CSV file was found inside:\n{model_dir}"
        )

    print("\n" + "=" * 100)
    print(f"Searching CSV files for model: {model_name}")
    print(f"Folder: {model_dir}")
    print(f"CSV files found: {len(csv_files)}")

    candidate_rows = []

    seed_candidates = [
        "seed", "random_seed", "run_seed", "training_seed", "train_seed"
    ]

    fold_candidates = [
        "fold", "fold_id", "cv_fold", "kfold", "outer_fold",
        "test_fold", "split", "split_id"
    ]

    all_metric_names = sorted(
        set(sum(METRIC_CANDIDATES.values(), []))
    )

    for csv_path in csv_files:
        try:
            df_head = pd.read_csv(csv_path, nrows=10)
            df_head = clean_columns(df_head)

            columns = df_head.columns.tolist()

            seed_col = find_first_existing_column(df_head, seed_candidates)
            fold_col = find_first_existing_column(df_head, fold_candidates)

            inferred_seed = infer_seed_from_path(csv_path)

            metric_cols = [
                col for col in all_metric_names
                if col in columns
            ]

            try:
                n_rows = len(pd.read_csv(csv_path))
            except Exception:
                n_rows = -1

            has_seed = seed_col is not None or inferred_seed is not None
            has_fold = fold_col is not None
            has_metric = len(metric_cols) > 0

            score = 0
            if has_seed:
                score += 10
            if has_fold:
                score += 10
            if has_metric:
                score += 10
            if os.path.basename(csv_path).lower() == "all_fold_results.csv":
                score += 5
            if "fold" in os.path.basename(csv_path).lower():
                score += 2
            if n_rows >= 50:
                score += 2

            candidate_rows.append({
                "csv_path": csv_path,
                "n_rows": n_rows,
                "seed_col": seed_col,
                "fold_col": fold_col,
                "inferred_seed": inferred_seed,
                "metric_cols": ", ".join(metric_cols),
                "score": score,
                "columns": ", ".join(columns),
            })

        except Exception as e:
            candidate_rows.append({
                "csv_path": csv_path,
                "n_rows": -1,
                "seed_col": None,
                "fold_col": None,
                "inferred_seed": None,
                "metric_cols": "",
                "score": -1,
                "columns": f"ERROR: {e}",
            })

    candidates_df = pd.DataFrame(candidate_rows)
    candidates_df = candidates_df.sort_values(
        ["score", "n_rows"],
        ascending=[False, False]
    ).reset_index(drop=True)

    report_path = os.path.join(
        OUT_DIR,
        f"csv_candidates_{model_name.replace(' ', '_').replace('/', '_')}.csv"
    )
    candidates_df.to_csv(report_path, index=False)

    print("\nCandidate CSV files:")
    display(candidates_df[[
        "csv_path", "n_rows", "seed_col", "fold_col",
        "inferred_seed", "metric_cols", "score"
    ]].head(20))

    valid = candidates_df[
        candidates_df["score"] >= 30
    ].copy()

    if len(valid) == 0:
        raise FileNotFoundError(
            f"No valid CSV file was found for {model_name}.\n"
            f"A valid CSV must contain seed, fold, and at least one metric column.\n"
            f"A diagnostic report was saved here:\n{report_path}"
        )

    selected_csv = valid.iloc[0]["csv_path"]

    print("\nSelected CSV file:")
    print(selected_csv)

    return selected_csv


def load_model_results(model_name, model_dir):
    """
    Load the most suitable CSV file for a given model.
    """
    csv_path = select_best_csv_for_model(model_name, model_dir)

    df = pd.read_csv(csv_path)
    df = clean_columns(df)

    seed_col = find_first_existing_column(
        df,
        ["seed", "random_seed", "run_seed", "training_seed", "train_seed"]
    )

    fold_col = find_first_existing_column(
        df,
        ["fold", "fold_id", "cv_fold", "kfold", "outer_fold",
         "test_fold", "split", "split_id"]
    )

    if fold_col is None:
        raise ValueError(
            f"No fold column was found for {model_name}.\n"
            f"CSV file used: {csv_path}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    if seed_col is None:
        inferred_seed = infer_seed_from_path(csv_path)

        if inferred_seed is None:
            raise ValueError(
                f"No seed column was found for {model_name}, and the seed could not be inferred from the path.\n"
                f"CSV file used: {csv_path}\n"
                f"Available columns: {df.columns.tolist()}"
            )

        df["seed"] = int(inferred_seed)
    else:
        df["seed"] = df[seed_col].astype(int)

    df["fold"] = df[fold_col].astype(int)

    df["pair_id"] = (
        df["seed"].astype(str)
        + "_fold_"
        + df["fold"].astype(str)
    )

    duplicated = df["pair_id"].duplicated().sum()

    if duplicated > 0:
        duplicated_rows = df[df["pair_id"].duplicated(keep=False)].sort_values("pair_id")

        dup_path = os.path.join(
            OUT_DIR,
            f"duplicated_pair_ids_{model_name.replace(' ', '_')}.csv"
        )

        duplicated_rows.to_csv(dup_path, index=False)

        raise ValueError(
            f"{model_name} has {duplicated} duplicated pair_id values.\n"
            f"Please inspect this file:\n{dup_path}"
        )

    print("\n" + "-" * 100)
    print(f"Model loaded: {model_name}")
    print(f"CSV file used: {csv_path}")
    print(f"Rows loaded: {len(df)}")
    print("Columns:")
    print(df.columns.tolist())

    return df


# ============================================================
# 5) DATA TABLES AND PERFORMANCE SUMMARY
# ============================================================

def make_wide_table(model_dfs, metric_name, metric_candidates):
    """
    Create a wide table:
    pair_id | seed | fold | Model 1 | Model 2 | ...

    Only seed-fold pairs present in all models are retained.
    """
    wide = None
    used_columns = {}

    for model_name, df in model_dfs.items():
        metric_col = find_first_existing_column(df, metric_candidates)

        if metric_col is None:
            print(
                f"\nMetric '{metric_name}' was skipped because it was not found in {model_name}."
            )
            print(f"Available columns in {model_name}:")
            print(df.columns.tolist())
            return None, None

        used_columns[model_name] = metric_col

        temp = df[["pair_id", "seed", "fold", metric_col]].copy()
        temp = temp.rename(columns={metric_col: model_name})

        if wide is None:
            wide = temp
        else:
            wide = pd.merge(
                wide,
                temp[["pair_id", model_name]],
                on="pair_id",
                how="inner"
            )

    model_names = list(model_dfs.keys())

    wide = wide.sort_values(["seed", "fold"]).reset_index(drop=True)
    wide = wide.dropna(subset=model_names)

    wide = normalize_wide_to_percent(wide, model_names)

    return wide, used_columns


def make_performance_summary_table(model_dfs, metric_candidates_dict):
    """
    Create a performance summary table using mean ± standard deviation
    for all available metrics.
    """
    rows = []

    for model_name, df in model_dfs.items():
        row = {"Model": model_name}

        for metric_name, candidates in metric_candidates_dict.items():
            metric_col = find_first_existing_column(df, candidates)

            if metric_col is None:
                row[f"{metric_name}_mean_%"] = np.nan
                row[f"{metric_name}_std_%"] = np.nan
                row[f"{metric_name}_latex"] = ""
                continue

            values = df[metric_col].astype(float).values
            values = normalize_series_to_percent(values)

            mean_val = np.nanmean(values)
            std_val = np.nanstd(values, ddof=1)

            row[f"{metric_name}_mean_%"] = mean_val
            row[f"{metric_name}_std_%"] = std_val
            row[f"{metric_name}_latex"] = f"{mean_val:.2f} $\\pm$ {std_val:.2f}"

        rows.append(row)

    summary = pd.DataFrame(rows)

    if "accuracy_mean_%" in summary.columns:
        summary = summary.sort_values(
            "accuracy_mean_%",
            ascending=False
        ).reset_index(drop=True)

    return summary


# ============================================================
# 6) STATISTICAL TESTS
# ============================================================

def pairwise_wilcoxon_all_models(wide, metric_name):
    """
    Run paired Wilcoxon signed-rank tests for all model pairs.
    """
    model_names = [
        c for c in wide.columns
        if c not in ["pair_id", "seed", "fold"]
    ]

    rows = []

    for model_a, model_b in itertools.combinations(model_names, 2):
        x = wide[model_a].astype(float).values
        y = wide[model_b].astype(float).values

        diff = x - y
        n = len(diff)

        try:
            w_stat, p_value = wilcoxon(
                x,
                y,
                zero_method="wilcox",
                alternative="two-sided",
                method="auto"
            )
        except ValueError:
            w_stat, p_value = np.nan, np.nan

        rows.append({
            "metric": metric_name,
            "model_A": model_a,
            "model_B": model_b,
            "n_pairs": n,

            "model_A_mean_%": np.mean(x),
            "model_A_std_%": np.std(x, ddof=1),
            "model_B_mean_%": np.mean(y),
            "model_B_std_%": np.std(y, ddof=1),

            "mean_diff_A_minus_B_pp": np.mean(diff),
            "median_diff_A_minus_B_pp": np.median(diff),

            "A_wins": int(np.sum(diff > 0)),
            "B_wins": int(np.sum(diff < 0)),
            "ties": int(np.sum(diff == 0)),

            "wilcoxon_W": w_stat,
            "wilcoxon_p": p_value,
        })

    df_pairwise = pd.DataFrame(rows)

    df_pairwise["wilcoxon_p_holm"] = holm_bonferroni_adjust(
        df_pairwise["wilcoxon_p"].values
    )

    df_pairwise["significant_raw_0.05"] = (
        df_pairwise["wilcoxon_p"] < ALPHA
    )

    df_pairwise["significant_holm_0.05"] = (
        df_pairwise["wilcoxon_p_holm"] < ALPHA
    )

    return df_pairwise


def make_pvalue_matrix(df_pairwise, p_col):
    """
    Create a model-by-model p-value matrix.
    """
    models = sorted(
        set(df_pairwise["model_A"].tolist())
        | set(df_pairwise["model_B"].tolist())
    )

    mat = pd.DataFrame(np.nan, index=models, columns=models)

    for model in models:
        mat.loc[model, model] = 1.0

    for _, row in df_pairwise.iterrows():
        a = row["model_A"]
        b = row["model_B"]
        p = row[p_col]

        mat.loc[a, b] = p
        mat.loc[b, a] = p

    return mat


# ============================================================
# 7) ARTICLE-STYLE STATISTICAL TABLE
# ============================================================

def make_article_stat_table(wide, df_pairwise, target_model=TARGET_MODEL):
    """
    Create the article-style statistical table:
    Model | Avg. Rank | Holm-corrected p-value vs EEG-Former | Significant
    """
    model_names = [
        c for c in wide.columns
        if c not in ["pair_id", "seed", "fold"]
    ]

    if target_model not in model_names:
        raise ValueError(
            f"The target model '{target_model}' is not present in the wide table.\n"
            f"Available models: {model_names}"
        )

    ranks = wide[model_names].rank(
        axis=1,
        ascending=False,
        method="average"
    )

    avg_ranks = ranks.mean(axis=0)

    rows = []

    for model in model_names:

        if model == target_model:
            raw_p = np.nan
            holm_p = np.nan
            significant = "---"
        else:
            mask = (
                ((df_pairwise["model_A"] == target_model) & (df_pairwise["model_B"] == model))
                |
                ((df_pairwise["model_A"] == model) & (df_pairwise["model_B"] == target_model))
            )

            if mask.sum() == 0:
                raw_p = np.nan
                holm_p = np.nan
                significant = "NA"
            else:
                raw_p = df_pairwise.loc[mask, "wilcoxon_p"].values[0]
                holm_p = df_pairwise.loc[mask, "wilcoxon_p_holm"].values[0]
                significant = "Yes" if holm_p < ALPHA else "No"

        rows.append({
            "Model": model,
            "Avg. Rank": avg_ranks[model],
            f"Wilcoxon p vs {target_model}": raw_p,
            f"Holm p vs {target_model}": holm_p,
            "Holm p formatted": format_p_display(holm_p),
            f"Significant vs {target_model}": significant,
        })

    article_table = pd.DataFrame(rows)
    article_table = article_table.sort_values("Avg. Rank").reset_index(drop=True)

    return article_table


def make_article_latex_table(article_table, target_model=TARGET_MODEL):
    """
    Generate the LaTeX article table using the full page width.
    The table content is saved to a file but not printed.
    """
    df = article_table.copy()

    latex_rows = []

    for _, row in df.iterrows():
        model = row["Model"]
        avg_rank = row["Avg. Rank"]
        holm_p = row[f"Holm p vs {target_model}"]
        sig = row[f"Significant vs {target_model}"]

        if model == target_model:
            model_tex = f"\\textbf{{{model}}}"
            rank_tex = f"$\\mathbf{{{avg_rank:.2f}}}$"
            p_tex = "---"
            sig_tex = "---"
        else:
            model_tex = model
            rank_tex = f"${avg_rank:.2f}$"
            p_tex = format_p_latex(holm_p)
            sig_tex = sig

        latex_rows.append(
            f"{model_tex} & {rank_tex} & {p_tex} & {sig_tex} \\\\"
        )

    latex = r"""
\begin{table}[H]
\centering
\renewcommand{\arraystretch}{1.15}
\begin{tabularx}{\textwidth}{>{\raggedright\arraybackslash}X c c c}
\toprule
\textbf{Model} 
& \shortstack{\textbf{Avg.}\\\textbf{Rank}} 
& \shortstack{\textbf{Holm-corrected}\\\textbf{$p$-value}} 
& \shortstack{\textbf{Significant vs.}\\\textbf{EEG-Former}} \\
\midrule
""" + "\n".join(latex_rows) + r"""
\bottomrule
\end{tabularx}
\caption{Statistical comparison of models under the repeated-seed subject-independent evaluation protocol. Lower average rank indicates better performance. Statistical significance was assessed using the Wilcoxon signed-rank post hoc test with Holm correction against EEG-Former at $\alpha=0.05$.}
\label{tab:statistical_comparison_eegformer}
\end{table}
"""

    return latex


# ============================================================
# 8) LOAD ALL MODELS
# ============================================================

model_dfs = {}

for model_name, model_dir in MODEL_PATHS.items():
    model_dfs[model_name] = load_model_results(model_name, model_dir)


# ============================================================
# 9) GENERAL PERFORMANCE SUMMARY TABLE
# ============================================================

performance_summary = make_performance_summary_table(
    model_dfs=model_dfs,
    metric_candidates_dict=METRIC_CANDIDATES
)

performance_summary_path = os.path.join(
    OUT_DIR,
    "performance_summary_all_metrics_mean_std.csv"
)

performance_summary.to_csv(performance_summary_path, index=False)

print("\n" + "=" * 100)
print("GENERAL PERFORMANCE SUMMARY: MEAN ± STD")
print("=" * 100)
display(performance_summary)


# ============================================================
# 10) FRIEDMAN + WILCOXON FOR ACCURACY
# ============================================================

all_pairwise_results = []
friedman_results = []
article_tables = {}

for metric_name in STATISTICAL_METRICS:

    candidates = METRIC_CANDIDATES[metric_name]

    print("\n" + "#" * 100)
    print(f"MAIN STATISTICAL METRIC: {metric_name}")
    print("#" * 100)

    wide, used_columns = make_wide_table(
        model_dfs=model_dfs,
        metric_name=metric_name,
        metric_candidates=candidates
    )

    if wide is None:
        print(f"Metric skipped: {metric_name}")
        continue

    model_names = list(MODEL_PATHS.keys())

    print("\nMetric columns used by model:")
    print(used_columns)

    print(f"\nNumber of complete paired observations found: {len(wide)}")

    if len(wide) != EXPECTED_N_PAIRS:
        print(
            "\nWARNING:"
            f"\nExpected {EXPECTED_N_PAIRS} complete paired observations "
            "(10 seeds x 5 folds), "
            f"but found {len(wide)}."
        )
        print(
            "Please check whether all models contain the exact same seeds and folds."
        )

    print("\nFirst rows of the wide table:")
    display(wide.head())

    wide_path = os.path.join(
        OUT_DIR,
        f"wide_{metric_name}_all_models_common_seed_fold_pairs.csv"
    )
    wide.to_csv(wide_path, index=False)

    # -------------------------------
    # Global Friedman test
    # -------------------------------
    friedman_inputs = [
        wide[model].astype(float).values
        for model in model_names
    ]

    friedman_stat, friedman_p = friedmanchisquare(*friedman_inputs)

    friedman_results.append({
        "metric": metric_name,
        "n_pairs": len(wide),
        "n_models": len(model_names),
        "friedman_stat": friedman_stat,
        "friedman_p": friedman_p,
        "friedman_p_formatted": format_p_display(friedman_p),
        "significant_0.05": friedman_p < ALPHA,
    })

    print("\nGLOBAL FRIEDMAN TEST")
    print(f"Statistic = {friedman_stat:.6f}")
    print(f"p-value   = {friedman_p:.2e}")

    # -------------------------------
    # Paired Wilcoxon tests, all vs all
    # -------------------------------
    df_pairwise = pairwise_wilcoxon_all_models(
        wide=wide,
        metric_name=metric_name
    )

    all_pairwise_results.append(df_pairwise)

    print("\nPAIRED WILCOXON TESTS: ALL MODEL PAIRS")
    display(df_pairwise)

    pairwise_path = os.path.join(
        OUT_DIR,
        f"pairwise_wilcoxon_all_models_{metric_name}.csv"
    )
    df_pairwise.to_csv(pairwise_path, index=False)

    # Raw p-value matrix
    p_matrix = make_pvalue_matrix(df_pairwise, "wilcoxon_p")
    p_matrix_path = os.path.join(
        OUT_DIR,
        f"matrix_wilcoxon_p_raw_{metric_name}.csv"
    )
    p_matrix.to_csv(p_matrix_path)

    # Holm-corrected p-value matrix
    p_holm_matrix = make_pvalue_matrix(df_pairwise, "wilcoxon_p_holm")
    p_holm_matrix_path = os.path.join(
        OUT_DIR,
        f"matrix_wilcoxon_p_holm_{metric_name}.csv"
    )
    p_holm_matrix.to_csv(p_holm_matrix_path)

    # -------------------------------
    # Article-style table
    # -------------------------------
    article_table = make_article_stat_table(
        wide=wide,
        df_pairwise=df_pairwise,
        target_model=TARGET_MODEL
    )

    article_tables[metric_name] = article_table

    article_table_path = os.path.join(
        OUT_DIR,
        f"article_statistical_table_{metric_name}_vs_EEGFormer.csv"
    )
    article_table.to_csv(article_table_path, index=False)

    print("\nARTICLE-STYLE STATISTICAL TABLE")

    article_table_display = article_table[[
        "Model",
        "Avg. Rank",
        "Holm p formatted",
        f"Significant vs {TARGET_MODEL}"
    ]].copy()

    article_table_display = article_table_display.rename(columns={
        "Holm p formatted": f"Holm-corrected p-value vs {TARGET_MODEL}",
        f"Significant vs {TARGET_MODEL}": f"Significant vs {TARGET_MODEL}",
    })

    display(article_table_display)

    # Generate and save LaTeX table, but do not print it
    latex_table = make_article_latex_table(
        article_table=article_table,
        target_model=TARGET_MODEL
    )

    latex_table_path = os.path.join(
        OUT_DIR,
        f"latex_article_statistical_table_{metric_name}_vs_EEGFormer.tex"
    )

    with open(latex_table_path, "w", encoding="utf-8") as f:
        f.write(latex_table)


# ============================================================
# 11) SAVE GENERAL SUMMARIES
# ============================================================

df_friedman = pd.DataFrame(friedman_results)

friedman_path = os.path.join(
    OUT_DIR,
    "friedman_global_tests_statistical_metrics.csv"
)
df_friedman.to_csv(friedman_path, index=False)

if len(all_pairwise_results) > 0:
    df_all_pairwise = pd.concat(all_pairwise_results, ignore_index=True)
else:
    df_all_pairwise = pd.DataFrame()

pairwise_all_path = os.path.join(
    OUT_DIR,
    "pairwise_wilcoxon_all_models_statistical_metrics.csv"
)
df_all_pairwise.to_csv(pairwise_all_path, index=False)


# ============================================================
# 12) FINAL SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("FRIEDMAN SUMMARY")
print("=" * 100)
display(df_friedman)

print("\n" + "=" * 100)
print("PAIRWISE WILCOXON SUMMARY")
print("=" * 100)
display(df_all_pairwise)

print("\n" + "=" * 100)
print("FILES SAVED TO:")
print("=" * 100)
print(OUT_DIR)

print("\nMain output files:")
print(f"- Performance summary mean ± std: {performance_summary_path}")
print(f"- Global Friedman test: {friedman_path}")
print(f"- Pairwise Wilcoxon tests: {pairwise_all_path}")

for metric_name in article_tables.keys():
    print(
        f"- Article-style CSV table ({metric_name}): "
        f"{os.path.join(OUT_DIR, f'article_statistical_table_{metric_name}_vs_EEGFormer.csv')}"
    )
    print(
        f"- Article-style LaTeX table ({metric_name}): "
        f"{os.path.join(OUT_DIR, f'latex_article_statistical_table_{metric_name}_vs_EEGFormer.tex')}"
    )

print("\nDONE.")


Searching CSV files for model: CNN-LSTM EEGNet
Folder: /kaggle/input/datasets/alejandragomezr/seed-cnn-lstm/results_cnn_lstm_eegnet_10seeds
CSV files found: 24

Candidate CSV files:


,csv_path,n_rows,seed_col,fold_col,inferred_seed,metric_cols,score
0,/kaggle/input/datasets/alejandragomezr/seed-cn...,50,seed,fold,NaN,"accuracy, auc, balanced_acc, f1, precision, re...",39
1,/kaggle/input/datasets/alejandragomezr/seed-cn...,5,seed,fold,123.0,"accuracy, auc, balanced_acc, f1, precision, re...",32
2,/kaggle/input/datasets/alejandragomezr/seed-cn...,5,seed,fold,2024.0,"accuracy, auc, balanced_acc, f1, precision, re...",32
3,/kaggle/input/datasets/alejandragomezr/seed-cn...,5,seed,fold,2718.0,"accuracy, auc, balanced_acc, f1, precision, re...",32
4,/kaggle/input/datasets/alejandragomezr/seed-cn...,5,seed,fold,314.0,"accuracy, auc, balanced_acc, f1, precision, re...",32
5,/kaggle/input/datasets/alejandragomezr/seed-cn...,5,seed,fold,42.0,"accuracy, auc, balanced_acc, f1, precision, re...",32
6,/kaggle/input/datasets/alejandragomezr/seed-cn...,5,seed,fold,456.0,"accuracy, auc, balanced_acc, f1, precision, re...",32
7,/kaggle/input/datasets/alejandragomezr/seed-cn...,5,seed,fold,5000.0,"accuracy, auc, balanced_acc, f1, precision, re...",32
8,/kaggle/input/datasets/alejandragomezr/seed-cn...,5,seed,fold,7.0,"accuracy, auc, balanced_acc, f1, precision, re...",32
9,/kaggle/input/datasets/alejandragomezr/seed-cn...,5,seed,fold,789.0,"accuracy, auc, balanced_acc, f1, precision, re...",32



Selected CSV file:
/kaggle/input/datasets/alejandragomezr/seed-cnn-lstm/results_cnn_lstm_eegnet_10seeds/all_fold_results.csv

----------------------------------------------------------------------------------------------------
Model loaded: CNN-LSTM EEGNet
CSV file used: /kaggle/input/datasets/alejandragomezr/seed-cnn-lstm/results_cnn_lstm_eegnet_10seeds/all_fold_results.csv
Rows loaded: 50
Columns:
['seed', 'fold', 'accuracy', 'balanced_acc', 'recall', 'precision', 'f1', 'auc', 'model_args', 'compile_args', 'pair_id']

Searching CSV files for model: T-GARNet
Folder: /kaggle/input/datasets/alejandragomezr/seed-danna/results_tgarnet_10seeds
CSV files found: 23

Candidate CSV files:


,csv_path,n_rows,seed_col,fold_col,inferred_seed,metric_cols,score
0,/kaggle/input/datasets/alejandragomezr/seed-da...,50,seed,fold,NaN,"accuracy, auc, balanced_acc, f1",39
1,/kaggle/input/datasets/alejandragomezr/seed-da...,5,seed,fold,123.0,"accuracy, auc, balanced_acc, f1",32
2,/kaggle/input/datasets/alejandragomezr/seed-da...,5,seed,fold,2024.0,"accuracy, auc, balanced_acc, f1",32
3,/kaggle/input/datasets/alejandragomezr/seed-da...,5,seed,fold,2718.0,"accuracy, auc, balanced_acc, f1",32
4,/kaggle/input/datasets/alejandragomezr/seed-da...,5,seed,fold,314.0,"accuracy, auc, balanced_acc, f1",32
5,/kaggle/input/datasets/alejandragomezr/seed-da...,5,seed,fold,42.0,"accuracy, auc, balanced_acc, f1",32
6,/kaggle/input/datasets/alejandragomezr/seed-da...,5,seed,fold,456.0,"accuracy, auc, balanced_acc, f1",32
7,/kaggle/input/datasets/alejandragomezr/seed-da...,5,seed,fold,5000.0,"accuracy, auc, balanced_acc, f1",32
8,/kaggle/input/datasets/alejandragomezr/seed-da...,5,seed,fold,7.0,"accuracy, auc, balanced_acc, f1",32
9,/kaggle/input/datasets/alejandragomezr/seed-da...,5,seed,fold,789.0,"accuracy, auc, balanced_acc, f1",32



Selected CSV file:
/kaggle/input/datasets/alejandragomezr/seed-danna/results_tgarnet_10seeds/all_fold_results.csv

----------------------------------------------------------------------------------------------------
Model loaded: T-GARNet
CSV file used: /kaggle/input/datasets/alejandragomezr/seed-danna/results_tgarnet_10seeds/all_fold_results.csv
Rows loaded: 50
Columns:
['seed', 'fold', 'accuracy', 'balanced_acc', 'f1', 'auc', 'best_model_args', 'best_compile_args', 'pair_id']

Searching CSV files for model: EEGNet
Folder: /kaggle/input/datasets/alejandragomezr/seed-eegnet/results_eegnet_10seeds
CSV files found: 23

Candidate CSV files:


...


FRIEDMAN SUMMARY


,metric,n_pairs,n_models,friedman_stat,friedman_p,friedman_p_formatted,significant_0.05
0,accuracy,50,7,127.249147,4.876051e-25,4.88e-25,True



PAIRWISE WILCOXON SUMMARY


,metric,model_A,model_B,n_pairs,model_A_mean_%,model_A_std_%,model_B_mean_%,model_B_std_%,mean_diff_A_minus_B_pp,median_diff_A_minus_B_pp,A_wins,B_wins,ties,wilcoxon_W,wilcoxon_p,wilcoxon_p_holm,significant_raw_0.05,significant_holm_0.05
0,accuracy,CNN-LSTM EEGNet,T-GARNet,50,84.250000,7.352576,74.750000,8.265455,9.500000,8.333333,41,6,3,90.5,4.591080e-07,6.886621e-06,True,True
1,accuracy,CNN-LSTM EEGNet,EEGNet,50,84.250000,7.352576,80.666667,7.470653,3.583333,4.166667,30,10,10,203.5,5.314965e-03,3.188979e-02,True,True
2,accuracy,CNN-LSTM EEGNet,Multi-Stream Transformer,50,84.250000,7.352576,69.250000,7.936558,15.000000,16.666667,46,3,1,15.5,2.769392e-09,5.815724e-08,True,True
3,accuracy,CNN-LSTM EEGNet,ShallowConvNet,50,84.250000,7.352576,78.666667,9.435603,5.583333,4.166667,28,12,10,193.5,3.505362e-03,2.453754e-02,True,True
4,accuracy,CNN-LSTM EEGNet,IM--CBGT,50,84.250000,7.352576,70.583333,9.613049,13.666667,12.500000,43,4,3,59.0,8.656692e-08,1.471638e-06,True,True
5,accuracy,CNN-LSTM EEGNet,EEG-Former,50,84.250000,7.352576,84.166667,5.390110,0.083333,0.000000,20,17,13,346.0,9.332580e-01,1.000000e+00,False,False
6,accuracy,T-GARNet,EEGNet,50,74.750000,8.265455,80.666667,7.470653,-5.916667,-4.166667,11,38,1,300.5,1.837752e-03,1.470202e-02,True,True
7,accuracy,T-GARNet,Multi-Stream Transformer,50,74.750000,8.265455,69.250000,7.936558,5.500000,4.166667,34,9,7,181.0,4.117139e-04,4.117139e-03,True,True
8,accuracy,T-GARNet,ShallowConvNet,50,74.750000,8.265455,78.666667,9.435603,-3.916667,-4.166667,15,28,7,296.0,3.186618e-02,1.274647e-01,True,False
9,accuracy,T-GARNet,IM--CBGT,50,74.750000,8.265455,70.583333,9.613049,4.166667,8.333333,30,15,5,279.0,6.939257e-03,3.469629e-02,True,True



FILES SAVED TO:
/kaggle/working/statistical_tests_EEGFormer_article

Main output files:
- Performance summary mean ± std: /kaggle/working/statistical_tests_EEGFormer_article/performance_summary_all_metrics_mean_std.csv
- Global Friedman test: /kaggle/working/statistical_tests_EEGFormer_article/friedman_global_tests_statistical_metrics.csv
- Pairwise Wilcoxon tests: /kaggle/working/statistical_tests_EEGFormer_article/pairwise_wilcoxon_all_models_statistical_metrics.csv
- Article-style CSV table (accuracy): /kaggle/working/statistical_tests_EEGFormer_article/article_statistical_table_accuracy_vs_EEGFormer.csv
- Article-style LaTeX table (accuracy): /kaggle/working/statistical_tests_EEGFormer_article/latex_article_statistical_table_accuracy_vs_EEGFormer.tex

DONE.


In [2]:
import os
import numpy as np
import pandas as pd

from scipy.stats import ttest_rel, wilcoxon, t

# ============================================================
# 1) PATHS
# ============================================================

PATH_EEGFORMER = "/kaggle/input/datasets/alejandragomezr/seed-eeg-former/results_ourmodel_10seeds"
PATH_CNN_LSTM = "/kaggle/input/datasets/alejandragomezr/seed-cnn-lstm/results_cnn_lstm_eegnet_10seeds"

CSV_EEGFORMER = os.path.join(PATH_EEGFORMER, "all_fold_results.csv")
CSV_CNN_LSTM = os.path.join(PATH_CNN_LSTM, "all_fold_results.csv")

OUT_DIR = "/kaggle/working/stability_tests_eegformer_vs_cnn_lstm"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# 2) HELPER FUNCTIONS
# ============================================================

def clean_columns(df):
    """
    Clean column names to make them easier to detect.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df


def find_metric_column(df, possible_names, model_name):
    """
    Find the first available metric column among a list of possible names.
    """
    for name in possible_names:
        if name in df.columns:
            return name
    
    raise ValueError(
        f"None of these columns were found for {model_name}: {possible_names}\n"
        f"Available columns: {df.columns.tolist()}"
    )


def load_fold_results(csv_path, model_name):
    """
    Load fold-level results and create a paired identifier using seed and fold.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"The file does not exist: {csv_path}")
    
    df = pd.read_csv(csv_path)
    df = clean_columns(df)
    
    for col in ["seed", "fold"]:
        if col not in df.columns:
            raise ValueError(
                f"Column '{col}' was not found in {model_name}. "
                f"Available columns: {df.columns.tolist()}"
            )
    
    df["seed"] = df["seed"].astype(int)
    df["fold"] = df["fold"].astype(int)
    
    df["pair_id"] = (
        df["seed"].astype(str)
        + "_fold_"
        + df["fold"].astype(str)
    )
    
    return df


def scale_to_percent(x):
    """
    Convert values to percentage if they are in the [0, 1] scale.
    If values are already in the [0, 100] scale, keep them unchanged.
    """
    x = np.asarray(x, dtype=float)
    if np.nanmax(x) <= 1.5:
        return x * 100
    return x


def bootstrap_std_ratio(x, y, n_boot=20000, random_state=42):
    """
    Estimate a bootstrap confidence interval for the ratio of standard deviations.

    Parameters
    ----------
    x : array-like
        EEG-Former values in percentage.
    y : array-like
        CNN-LSTM EEGNet values in percentage.

    Returns
    -------
    dict
        Bootstrap estimates for std_x / std_y and std_x - std_y.

    Interpretation
    --------------
    If the confidence interval for std_x / std_y is below 1,
    EEG-Former is more stable than CNN-LSTM EEGNet.
    """
    rng = np.random.default_rng(random_state)
    n = len(x)
    
    ratios = []
    diff_stds = []
    
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        xb = x[idx]
        yb = y[idx]
        
        std_x = np.std(xb, ddof=1)
        std_y = np.std(yb, ddof=1)
        
        if std_y > 0:
            ratios.append(std_x / std_y)
            diff_stds.append(std_x - std_y)
    
    ratios = np.asarray(ratios)
    diff_stds = np.asarray(diff_stds)
    
    return {
        "std_ratio": np.std(x, ddof=1) / np.std(y, ddof=1),
        "std_ratio_ci95_low": np.percentile(ratios, 2.5),
        "std_ratio_ci95_high": np.percentile(ratios, 97.5),
        "std_diff": np.std(x, ddof=1) - np.std(y, ddof=1),
        "std_diff_ci95_low": np.percentile(diff_stds, 2.5),
        "std_diff_ci95_high": np.percentile(diff_stds, 97.5),
    }


def noninferiority_paired_t(diff, margin_pp=2.0):
    """
    Paired non-inferiority t-test.

    Parameters
    ----------
    diff : array-like
        Difference in percentage points:
        EEG-Former - CNN-LSTM EEGNet.
    margin_pp : float
        Non-inferiority margin in percentage points.

    Hypotheses
    ----------
    H0: mean_diff <= -margin_pp
    H1: mean_diff >  -margin_pp

    Interpretation
    --------------
    If p < 0.05, EEG-Former is non-inferior to CNN-LSTM EEGNet
    using the selected margin.
    """
    n = len(diff)
    mean_diff = np.mean(diff)
    sd_diff = np.std(diff, ddof=1)
    se = sd_diff / np.sqrt(n)
    
    t_stat = (mean_diff + margin_pp) / se
    p_value = 1 - t.cdf(t_stat, df=n - 1)
    
    ci90_low, ci90_high = t.interval(
        confidence=0.90,
        df=n - 1,
        loc=mean_diff,
        scale=se
    )
    
    return {
        "margin_pp": margin_pp,
        "mean_diff_pp": mean_diff,
        "t_noninferiority": t_stat,
        "p_noninferiority": p_value,
        "ci90_low_pp": ci90_low,
        "ci90_high_pp": ci90_high,
        "noninferior_alpha_0_05": p_value < 0.05,
    }


def stability_wilcoxon_abs_deviation(x, y, center="median"):
    """
    Compare model stability using absolute deviations.

    Parameters
    ----------
    x : array-like
        EEG-Former values in percentage.
    y : array-like
        CNN-LSTM EEGNet values in percentage.
    center : str
        Center used to compute deviations: 'median' or 'mean'.

    Hypotheses
    ----------
    H0: absolute deviations are similar.
    H1: EEG-Former has smaller absolute deviations.

    Interpretation
    --------------
    The alternative is 'less' because we test whether
    EEG-Former deviations are smaller than CNN-LSTM EEGNet deviations.
    """
    if center == "median":
        cx = np.median(x)
        cy = np.median(y)
    elif center == "mean":
        cx = np.mean(x)
        cy = np.mean(y)
    else:
        raise ValueError("center must be either 'median' or 'mean'")
    
    dev_x = np.abs(x - cx)
    dev_y = np.abs(y - cy)
    
    try:
        w_stat, p_value = wilcoxon(
            dev_x,
            dev_y,
            alternative="less",
            zero_method="wilcox"
        )
    except ValueError:
        w_stat, p_value = np.nan, np.nan
    
    return {
        "center": center,
        "EEGFormer_mean_abs_dev": np.mean(dev_x),
        "CNNLSTM_mean_abs_dev": np.mean(dev_y),
        "EEGFormer_median_abs_dev": np.median(dev_x),
        "CNNLSTM_median_abs_dev": np.median(dev_y),
        "wilcoxon_abs_dev_W": w_stat,
        "wilcoxon_abs_dev_p_less": p_value,
        "EEGFormer_more_stable_alpha_0_05": p_value < 0.05,
    }


# ============================================================
# 3) LOAD RESULTS
# ============================================================

df_eeg = load_fold_results(CSV_EEGFORMER, "EEG-Former")
df_cnn = load_fold_results(CSV_CNN_LSTM, "CNN-LSTM EEGNet")

metric_candidates = {
    "accuracy": ["accuracy", "acc", "test_accuracy", "test_acc"],
    "balanced_accuracy": ["balanced_acc", "balanced_accuracy", "bacc", "test_balanced_acc", "test_bacc"],
    "f1": ["f1", "f1_score", "test_f1"],
    "auc": ["auc", "roc_auc", "test_auc"],
}

all_results = []

for metric_name, candidates in metric_candidates.items():
    
    try:
        col_eeg = find_metric_column(df_eeg, candidates, "EEG-Former")
        col_cnn = find_metric_column(df_cnn, candidates, "CNN-LSTM EEGNet")
    except ValueError as e:
        print(f"\nSkipping {metric_name}:")
        print(e)
        continue
    
    eeg = df_eeg[["pair_id", "seed", "fold", col_eeg]].copy()
    eeg = eeg.rename(columns={col_eeg: "EEG-Former"})
    
    cnn = df_cnn[["pair_id", col_cnn]].copy()
    cnn = cnn.rename(columns={col_cnn: "CNN-LSTM EEGNet"})
    
    paired = pd.merge(eeg, cnn, on="pair_id", how="inner")
    paired = paired.sort_values(["seed", "fold"]).reset_index(drop=True)
    paired = paired.dropna(subset=["EEG-Former", "CNN-LSTM EEGNet"])
    
    x = scale_to_percent(paired["EEG-Former"].values)
    y = scale_to_percent(paired["CNN-LSTM EEGNet"].values)
    
    diff = x - y
    
    print("\n" + "=" * 90)
    print(f"METRIC: {metric_name}")
    print("=" * 90)
    print(f"Number of paired observations: {len(paired)}")
    
    if len(paired) != 50:
        print("WARNING: 50 complete paired observations were expected, but a different number was found.")
    
    # -------------------------------
    # Descriptive statistics
    # -------------------------------
    mean_x = np.mean(x)
    std_x = np.std(x, ddof=1)
    mean_y = np.mean(y)
    std_y = np.std(y, ddof=1)
    
    cv_x = std_x / mean_x
    cv_y = std_y / mean_y
    
    # -------------------------------
    # Paired Wilcoxon performance test
    # -------------------------------
    try:
        w_perf, p_perf = wilcoxon(x, y, alternative="two-sided")
    except ValueError:
        w_perf, p_perf = np.nan, np.nan
    
    # -------------------------------
    # Non-inferiority analysis
    # Change margin_pp if you want to test 1, 1.5, 2, or 2.5 pp.
    # -------------------------------
    noninf = noninferiority_paired_t(diff, margin_pp=2.0)
    
    # -------------------------------
    # Stability analysis using absolute deviations
    # -------------------------------
    stab_median = stability_wilcoxon_abs_deviation(x, y, center="median")
    stab_mean = stability_wilcoxon_abs_deviation(x, y, center="mean")
    
    # -------------------------------
    # Bootstrap analysis of the standard deviation ratio
    # -------------------------------
    boot = bootstrap_std_ratio(x, y, n_boot=20000, random_state=42)
    
    row = {
        "metric": metric_name,
        "n_pairs": len(paired),
        
        "EEGFormer_mean_%": mean_x,
        "EEGFormer_std_%": std_x,
        "EEGFormer_cv": cv_x,
        
        "CNNLSTM_mean_%": mean_y,
        "CNNLSTM_std_%": std_y,
        "CNNLSTM_cv": cv_y,
        
        "mean_diff_EEGFormer_minus_CNNLSTM_pp": np.mean(diff),
        
        "wilcoxon_performance_W": w_perf,
        "wilcoxon_performance_p": p_perf,
        
        **noninf,
        **{f"median_center_{k}": v for k, v in stab_median.items()},
        **{f"mean_center_{k}": v for k, v in stab_mean.items()},
        **boot,
    }
    
    all_results.append(row)
    
    display(pd.DataFrame([row]))

# ============================================================
# 4) SAVE RESULTS
# ============================================================

df_results = pd.DataFrame(all_results)

out_path = os.path.join(
    OUT_DIR,
    "eegformer_vs_cnn_lstm_noninferiority_and_stability.csv"
)

df_results.to_csv(out_path, index=False)

print("\nResults saved to:")
print(out_path)

display(df_results)


METRIC: accuracy
Number of paired observations: 50


,metric,n_pairs,EEGFormer_mean_%,EEGFormer_std_%,EEGFormer_cv,CNNLSTM_mean_%,CNNLSTM_std_%,CNNLSTM_cv,mean_diff_EEGFormer_minus_CNNLSTM_pp,wilcoxon_performance_W,...,mean_center_CNNLSTM_median_abs_dev,mean_center_wilcoxon_abs_dev_W,mean_center_wilcoxon_abs_dev_p_less,mean_center_EEGFormer_more_stable_alpha_0_05,std_ratio,std_ratio_ci95_low,std_ratio_ci95_high,std_diff,std_diff_ci95_low,std_diff_ci95_high
0,accuracy,50,84.166667,5.39011,0.064041,84.25,7.352576,0.087271,-0.083333,346.0,...,5.083333,492.0,0.079507,False,0.733091,0.576976,0.971892,-1.962466,-3.558733,-0.157115



METRIC: balanced_accuracy
Number of paired observations: 50


,metric,n_pairs,EEGFormer_mean_%,EEGFormer_std_%,EEGFormer_cv,CNNLSTM_mean_%,CNNLSTM_std_%,CNNLSTM_cv,mean_diff_EEGFormer_minus_CNNLSTM_pp,wilcoxon_performance_W,...,mean_center_CNNLSTM_median_abs_dev,mean_center_wilcoxon_abs_dev_W,mean_center_wilcoxon_abs_dev_p_less,mean_center_EEGFormer_more_stable_alpha_0_05,std_ratio,std_ratio_ci95_low,std_ratio_ci95_high,std_diff,std_diff_ci95_low,std_diff_ci95_high
0,balanced_accuracy,50,84.499145,5.453991,0.064545,84.411966,7.18673,0.085139,0.087179,406.5,...,4.195804,499.5,0.091184,False,0.758897,0.609662,0.997002,-1.732739,-3.261679,-0.016683



METRIC: f1
Number of paired observations: 50


,metric,n_pairs,EEGFormer_mean_%,EEGFormer_std_%,EEGFormer_cv,CNNLSTM_mean_%,CNNLSTM_std_%,CNNLSTM_cv,mean_diff_EEGFormer_minus_CNNLSTM_pp,wilcoxon_performance_W,...,mean_center_CNNLSTM_median_abs_dev,mean_center_wilcoxon_abs_dev_W,mean_center_wilcoxon_abs_dev_p_less,mean_center_EEGFormer_more_stable_alpha_0_05,std_ratio,std_ratio_ci95_low,std_ratio_ci95_high,std_diff,std_diff_ci95_low,std_diff_ci95_high
0,f1,50,85.538486,4.708079,0.05504,85.832697,6.115165,0.071245,-0.294211,380.0,...,4.351215,514.5,0.117309,False,0.769902,0.602447,1.001178,-1.407086,-2.719382,0.005515



METRIC: auc
Number of paired observations: 50


,metric,n_pairs,EEGFormer_mean_%,EEGFormer_std_%,EEGFormer_cv,CNNLSTM_mean_%,CNNLSTM_std_%,CNNLSTM_cv,mean_diff_EEGFormer_minus_CNNLSTM_pp,wilcoxon_performance_W,...,mean_center_CNNLSTM_median_abs_dev,mean_center_wilcoxon_abs_dev_W,mean_center_wilcoxon_abs_dev_p_less,mean_center_EEGFormer_more_stable_alpha_0_05,std_ratio,std_ratio_ci95_low,std_ratio_ci95_high,std_diff,std_diff_ci95_low,std_diff_ci95_high
0,auc,50,90.54245,5.236574,0.057836,91.978037,5.563834,0.060491,-1.435587,409.5,...,3.826159,580.0,0.289408,False,0.941181,0.760564,1.19605,-0.327259,-1.4977,0.881149



Results saved to:
/kaggle/working/stability_tests_eegformer_vs_cnn_lstm/eegformer_vs_cnn_lstm_noninferiority_and_stability.csv


,metric,n_pairs,EEGFormer_mean_%,EEGFormer_std_%,EEGFormer_cv,CNNLSTM_mean_%,CNNLSTM_std_%,CNNLSTM_cv,mean_diff_EEGFormer_minus_CNNLSTM_pp,wilcoxon_performance_W,...,mean_center_CNNLSTM_median_abs_dev,mean_center_wilcoxon_abs_dev_W,mean_center_wilcoxon_abs_dev_p_less,mean_center_EEGFormer_more_stable_alpha_0_05,std_ratio,std_ratio_ci95_low,std_ratio_ci95_high,std_diff,std_diff_ci95_low,std_diff_ci95_high
0,accuracy,50,84.166667,5.390110,0.064041,84.250000,7.352576,0.087271,-0.083333,346.0,...,5.083333,492.0,0.079507,False,0.733091,0.576976,0.971892,-1.962466,-3.558733,-0.157115
1,balanced_accuracy,50,84.499145,5.453991,0.064545,84.411966,7.186730,0.085139,0.087179,406.5,...,4.195804,499.5,0.091184,False,0.758897,0.609662,0.997002,-1.732739,-3.261679,-0.016683
2,f1,50,85.538486,4.708079,0.055040,85.832697,6.115165,0.071245,-0.294211,380.0,...,4.351215,514.5,0.117309,False,0.769902,0.602447,1.001178,-1.407086,-2.719382,0.005515
3,auc,50,90.542450,5.236574,0.057836,91.978037,5.563834,0.060491,-1.435587,409.5,...,3.826159,580.0,0.289408,False,0.941181,0.760564,1.196050,-0.327259,-1.497700,0.881149
